# Teresio - MinHashing

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

subset = "horoscope_full"
df = pd.read_csv(f'..\\data\\{subset}.csv')
df.head()

,ID,sign,category,date,horoscope
0,1,aries,general,20200617,"There's a great day ahead of you, Aries. You'l..."
1,2,aries,general,20200618,People will understand and appreciate your des...
2,3,aries,general,20200619,You are very interested in technological break...
3,4,aries,general,20200620,Stress from overwork could have you feeling we...
4,5,aries,general,20200621,This is a good day to stand up for yourself an...


In [3]:
df_general = df[df["category"] == "general"]

horoscope_general_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_general.iterrows()
}

In [4]:
import re

def normalize_text(text):
    text = text.lower()
    
    # 1. Remove punctuation and replace with a space
    text = re.sub(r"[^\w\s]", " ", text) # Removes everything that is not a word character or whitespace
    
    # 2. Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # 3. Return the list of words (tokens)
    return text.split(' ')


def shingle(q, text):
    newtext = normalize_text(text)
    shingles = []
    # The index must go from 0 to the last shingle index. 
    # The last shingle index is N (number of words) - q. 
    # We add +1 so it's not excluded by the Python for loop's range.
    for i in range(len(newtext) - q + 1): 
        # The join method combines elements of a list into a single string, 
        # separated by a space character (" "). In this case, 
        # the list elements go from index i to index i+q.
        shingles.append(" ".join(newtext[i:i + q])) 
    
    # Remove repetitions
    unique = set(shingles) 
    # Convert back to a list
    list_shingles = list(unique) 
    
    return list_shingles

In [5]:
import sys
import os
import mmh3


#################### Utilities ######################
#hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ mmh3.hash(e, seed)
	return val 

In [7]:
def signature(docs: dict, seedlist: list, q: int):
    
    
    doc_ids = list(docs.keys()) #estraggo id del dizionario dall dict, ora ho una lista con ID del dizionario
    k = len(seedlist) # k è il numero di funzioni hash
    
    
    shingle_to_docs = {} #preparo un dict vuoto
    
    # Ciclo iniziale per popolare l'indice e preparare gli shingles
    for doc_id, text in docs.items(): #associo in doc_id ID dei documenti e in text il testo dei documenti
        
        shingles_list = shingle(q, text) #preparo la lista di shingles, questo per un documento 
        for shingle_item in shingles_list:
          
            shingle_to_docs.setdefault(shingle_item, set()).add(doc_id)

        # il doppio ciclo mi serve per scorrere tutti i documenti e in ogni documento ogni shingle
        # ho la mia lista vuota a cui applico la funzione setdefault questa funzione prende la key (uno shingle) 
        # e la cerca all'interno del dizionario se c'è aggiunge un altro documento ID nella raccolta set dei value, 
        # se non c'è crea questa chiave e aggiunge L'ID del documento immediatamente all'interno di un oggetto set  
        # quello che ho alla fine è un grande dictionary con tutti gli shingles come key e i documenti dove sono presenti come value
         

    doc_signatures = {              #definisco il dizionario delle firme con id come key e una lista lunga quanto le funzioni hash con all'interno solo inf, quindi k inf
        doc_id: [float("inf")] * k  #[float("inf")] * k  quando moltiplichi una lista contenente un singolo elemento per un numero intero, il risultato 
                                    #è una nuova lista in cui quell'elemento è ripetuto k volte.
        for doc_id in doc_ids
    } # quindi ora ho un dict inizializzato con le key dei documenti e come value una lista lunga quanto il numero di f.ash di infinito
    
    # a questo punto ho un dict che ha come key gli shingles e come value la lista dove lo shingle si trova shingle_to_docs 
    # inoltre ho un dict doc_signatures che ha come key gli id dei documenti e come value invece una lista di infiniti tanti quante le funzioni hash 
    
    for shingle_item in shingle_to_docs.keys(): # prendi lo shingle (key) dalla lista di shingle (ciclato su tutti gli shingle)
        
        hash_values = [listhash(shingle_item, seed) for seed in seedlist] # prendo lo shingle e gli applico tutte l hash function e le salvo in una lista
        
        for doc_id in shingle_to_docs[shingle_item]: #scorri tutti gli ID dove trovi questo shingle. shingle_to_docs[shingle_item] è il value ovvero la lista ID dove puoi
                                                     #trovare lo shingle e scorri tutti gli ID
            current_signature = doc_signatures[doc_id]# la firma corrente è la prima volta tutti infiniti e poi si aggiorna con tutti i minimi
            for i in range(k):
                current_signature[i] = min(current_signature[i], hash_values[i])
                
    #prendo lo shingle e gli applico tutte le funzioni hash e le salvo in una lista che avrà lunghezza k
    #prendo la lista dei documenti dove questo shingle è presente
    # per ogni documento in cui è presente cerco la sua firma attuale
    # controllo funzione per funzione se l'ash che ho adesso è minore nel caso sostiuisco
    # quindi per ogni shingles, per ogni documento in cui è contenuto e per ogni funzione ash.            
    return doc_signatures

In [8]:
import numpy as np
def jaccard (doc_id_1: str, doc_id_2: str, doc_signatures: dict):
    sign1 = np.array(doc_signatures[doc_id_1])
    sign2 = np.array(doc_signatures[doc_id_2])
    matches = 0
    k = len(sign1)
    
    for i in range(k):
        if sign1[i] == sign2[i]:
            matches += 1
            
    similarity = matches / k
    return similarity

In [9]:
def similar(doc_signatures, threshold):
    list_keys = list(doc_signatures.keys()) # I save all the keys of the documents in a list
    similar_items = {} # empty dict to save the two ids and the similarity value
    
    # double for cycle with i less than j so as not to repeat the combinations
    for i in range (len(list_keys)-1): 
        for j in range (i + 1, len(list_keys)):
            similarity_score = jaccard(list_keys[i], list_keys[j], doc_signatures)
            if similarity_score >= threshold:
                # as key I put the two IDs of the documents, as value the similarity value
                similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                
    return similar_items

In [10]:
def lsh(signatures_dict, b, jaccard_threshold=0.5, seed=42):
    lsh_dict = {} # new dict that has as key the IDs and as value the hashes of the blocks
    for key, values in signatures_dict.items(): # for each item with its key value ID: signature
        blocks = np.split(np.array(values), b) # split the signature into blocks
        blocks_hash_values = [] # empty list for the new signature
        for aBlock in blocks: # for each block among the blocks
            band_bytes = aBlock.tobytes()
            # hash for each block until a list of hashes is created
            blocks_hash_values.append(mmh3.hash(band_bytes, seed)) 
        # in the dict I put the ID in the key and the new list of hashed blocks in the value
        lsh_dict[key] = blocks_hash_values 
        
    list_keys = list(lsh_dict.keys()) # I save the list of keys
    similar_items = {} # new dict
    
    for i in range (len(list_keys)-1):
        for j in range (i+1, len(list_keys)):
            # how many in common in the new list?
            common_values = np.intersect1d(lsh_dict[list_keys[i]], lsh_dict[list_keys[j]]) 
            
            # if at least one then they are candidates and I calculate them with jaccard
            if len(common_values) > 0: 
                # we found a candidate
                similarity_score = jaccard(list_keys[i], list_keys[j], signatures_dict)
                
                # if they exceed the threshold
                if similarity_score >= jaccard_threshold: 
                    # the key of similar items are the name of the two documents and the value is the similarity
                    similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                    
    return similar_items

In [ ]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_general_dict, seedlist, q)

# Find similar items using the LSH function

similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)


# ---------------- Stampa risultati ----------------
# Print results

for (id1, id2), sim in similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_general_dict[id1]}")
    print(f"Text2: {horoscope_general_dict[id2]}")
    print("------")